PyDara 0.6.0a2

آزمون زنده TSETMC، Windows، Python 3.14 و Pandas 3 در این محیط تأیید نشده‌اند. نمونه‌ها مصنوعی هستند.

## 1. Imports / ورود کتابخانه‌ها

In [ ]:
import pandas as pd
from pathlib import Path
from tempfile import TemporaryDirectory
import pydara
from pydara import tse
from pydara.tse.samples.market import sample_market_client
print(pydara.__version__)
assert hasattr(data, "get_market_watch")

## 2. Helpers

In [ ]:
def preview(frame):
    display(tse.select_columns(frame))
    schema = frame.attrs["pydara"]["schema_id"]
    issues = tse.validate_data(frame, schema=schema)
    assert issues.loc[issues.severity == "error"].empty, issues
    return issues

## 3. Settings

In [ ]:
SYMBOL = "فولاد"
INDEX = "32097828799138957"

## 4. Synthetic data, NOT live prices

In [ ]:
with sample_market_client() as client:
    watch = client.get_market_watch(include_order_book=True, include_client_types=True)
    people = client.get_client_type_history(SYMBOL, include_jdate=True)
    indices = client.get_indices(market=None)
    history = client.get_index_history(INDEX, include_jdate=True)
for frame in (watch, people, indices, history):
    preview(frame)

## 5. Ordered Pandas inputs

In [ ]:
with sample_market_client() as client:
    batch = client.get_client_type_history_many(pd.Series([SYMBOL, SYMBOL]), on_error="collect")
display(batch.requests)
display(batch.data)
assert len(batch.requests) == 2 and batch.errors.empty

## 6. Cache, no real HTTP, and CSV roundtrip

In [ ]:
with TemporaryDirectory(prefix="pydara-market-demo-") as folder:
    with sample_market_client(storage=True, data_dir=folder) as client:
        original = client.get_market_watch()
        client.flush(timeout=10)
    with sample_market_client(storage=True, data_dir=folder, mode="offline") as client:
        cached = client.get_market_watch(policy=tse.FetchPolicy.offline(allow_stale=True))
        assert client.request_count == 0
        print("Offline rows:", len(cached))
    path = Path(folder) / "market.csv"
    tse.save(original, path)
    restored = tse.load(path)
    pd.testing.assert_frame_equal(original, restored, check_dtype=True)
    print("CSV roundtrip OK")

## 7. Installed function contracts

In [ ]:
display(tse.api_reference("get_market_watch"))
display(tse.schema_info("client_types"))